## This Notebook will be used to make analytical operations with documented outcomes and insights.

We start by importing the different libraries and setting up a path finder for future functions imports


In [ ]:
#importing necessary libraries and setting up paths. do not update this cell. Run first before running any other cell. 
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

analytics_path = Path("../data/analytics")
session_path = (analytics_path / "session_analytics.parquet").as_posix()
visitor_path = (analytics_path / "visitor_analytics.parquet").as_posix()
question_path = (analytics_path / "question_analytics.parquet").as_posix()


This first operation highlights the campaigns overview and the outcomes distribution.

32.9% of sessions remain incomplete.  
We also conclude the outcomes distribution: 70% represent no current indication; 20% are possibly at risk; 10% already are diagnosed with diabetes.  
We also have a number of 2800 returning visitors that have had multiple sessions.


In [ ]:
from src.analytics.campaign_overview import (
    get_campaign_overview,
    get_outcome_distribution
)

display(get_campaign_overview(session_path))
display(get_outcome_distribution(session_path))

This operation uses the results of the previous one to determine the starting rate for each visitor (takes account of visitors that have started multiple sessions) which is 66.17%.  
Only 66.17% of visitors start the questionnaire.

In [ ]:
from src.analytics.campaign_overview import get_visitor_start_rate

display(get_visitor_start_rate(session_path))

This next operation highlights the audience differences in outcomes. 


Outcomes are really similar across all genders but differ a lot across age groups which is expected. The average age of each outcome being: 51 diagnosed; 50 at risk; 41.4 no current indication.


Relatively few visitors ommited sharing personnal data which is not a problem, the number is not consequent to influence data.  
A returning visitor factor may not be a poroblem since the completion rate across all ages is the same, which is around 0.56 session completed per person.


The campaign reached visitors across all age groups, with 35-44 and 45-54 being the most represented as they form half of the completed sessions.  
No real influence detected regarding the region. Ages across regions are the same and we can say the same thing about genders, some regions have a higher female percentage but the number of individuals is not consequent to conclude anything.




In [ ]:
from src.analytics.audience import (
    outcome_by_age,
    outcome_by_gender,
    outcome_by_region,
    age_by_outcome,
    missing_demographics,
    median_age_by_region,
    visitors_by_age_group,
    completed_sessions_by_age,
    gender_by_region
)

display(outcome_by_age(session_path))
display(outcome_by_gender(session_path))
display(outcome_by_region(session_path))
display(gender_by_region(visitor_path))
display(age_by_outcome(session_path))
display(visitors_by_age_group(visitor_path))
display(missing_demographics(visitor_path))
display(median_age_by_region(visitor_path))
display(completed_sessions_by_age(visitor_path))

This next operation hightlights campaigns performance to analyse which campaign succeeded the most in term of completion and user interaction.  

After comparing different campaigns all of them had they same number of people except for "diabetes_retargeting" which had significantly less visitors but overperformed relatively to others regarding the sessions started and completed. Also most of the sessions had no campaign name which is unfortunate.  
All of the campaigns had almost the same outcome distribution except for "diabetes_45plus" and "diabetes_social_awareness" which suggest further analysis.  
Further analysis reveals that "diabetes_45plus" had a higher average visitor age while "diabetes_social_awareness" presents a lower average visitor age, which explains the difference in outcomes provides by each one.  

For the acquisition sources, social media sources underperform but the results cannot be interpreted since the numbrer of sessions is little compared to the whole study case. In the otherhand "google-organic" and "google-ads" perform well and have the highest started sessions count across all sources, "chatgpt", "refferal" and "email" detain the most efficient results, relatively high session count, higher start rate and excellent completion rate. acquisition source does not seem to have any link of influence with the outcomes  

We notice that mobile users visit the most the campaigns but it comes at a cost, mobile users have the least completion rate across all devices. In the other hand Desktop as a device type hold a nice balance between sessions and completion, tablet is not really used across visitors but it still hold a respectable completion rate. Further analysis should be done to inspect if there is any correlation between devices used and outcomes (even if there is, it does not mean that there is any causation).  

Further analysis concludes that there is no correlation between devices used and outcomes provided.

In [ ]:
from src.analytics.campaign_performance import (
    performance_by,
    outcomes_by,
    age_by_campaign,
    device_by_source,
    outcomes_by_device,
)

display(performance_by(session_path, "campaign_name"))
display(outcomes_by(session_path, "campaign_name"))
display(performance_by(session_path, "acquisition_source"))
display(outcomes_by(session_path, "acquisition_source"))
display(performance_by(session_path, "device_type"))
display(device_by_source(session_path))
display(age_by_campaign(session_path))
display(outcomes_by_device(session_path))

This next operation will highlight information regarding the questionnaire and how visitors interact with it.  

We notice that most of the questions have high answer rates and little dropouts except for question 4 which has a 13.75% dropout rate, it is really high and be explained with different possibilities. we'll just make a note of it for now since i can't really explain if the reason comes from the question itself or from the questionnaires structure.  
And regarding the questionnaire structure, questions with possible "not sure" or "prefer not to say" answer have on average 8.5% of said answer, which is not bad, the questions are well structured in a way that they don't make the visitor confused or unconfortable, however the fifth question is in my opinion very confusing for people with an already given diagnosis, since they would not know how to answer it, they may understand that since they already have a diagnosis then they shoudl answer "no" or "yes" since they do have high blood sugar but got a diagnosis, i recommend that for this question we restructure answers only for people who already answer "yes" for the first question: "yes, before my diagnosis", "no, never before my diagnosis"

We cannot use questionnaire outcomes to correlate answers with real medical outcomes since the outcomes provided are results of the same questionnaire answers, so in this analysis the most important data for me will be the one of people who already had a diagnosis and then correlate their given answers with risks and reccurency.

After further analysis, people with an already given diagnosis tend to be 10% more positive to symptoms such as unusual thirst and frequent urination, while people at rist present 40% more positive answers to that same given symptom.  
Data about the physical activity question seems to be well rounded for people already diagnosed with type 2 diabetes, it could be explained by the fast that people who already received a diagnosis are more recommended to exercise for their own benefits by their health professionals. people at risk tend to be have a more sedentary lifestyle (58%) than people who aren't (13%) which is expected.
Only 32% of diagnosed people report a parent or sibling whit diagnosed type 2 diabetes which is twice as much as people with no current indicators, people at risk report twice as much as diagnosed people: 63%. I can't really interprete this data, i need more specifications about rankins, what makes a person at risk, what criteria from the questionnaire are used as base for an outcome providing, which will be my next step, try to deduct the scale used.  

After answers tendencies analysis i deduct that for a negative outcome to be provided at least 3 of these questions must get a positive answer:  
positive symptoms such as thirst and frequent urination .  
a parent or siblin with diagnosed type 2 diabetes.  
No physical activity.  
High blood sugar levels.  
Again the i have the same questions, at was stage is a symptom really considered a risk because we notice that people with diagnosis answer on average less positive to this scale than people at risk.


In [ ]:
from src.analytics.questionnaire_analysis import (
    question_performance,
    responses_by_outcome,
    prefer_not_to_say
)

display(question_performance(question_path))
display(responses_by_outcome(question_path))
display(prefer_not_to_say(question_path))

This next operation will leand to a further more personal analytical insights, it might provide me correlations between events and might as well have no relevant outcomes.  

For the first analysis (cell 1) i noticed that people between 35 and 44 years old with a possible risk outcome tend to stay awake late at night which could be correlated with a sedentary lifestyle, the data is little so i cannot conclude with anything, this is just an insight with no real value.  

The second analysis (cell 2) helped me notice a gain in popularity for the campaign, the compared results of June and July show a difference of 2000 new visitors, the compaign recoded 9500 new visitors for the month of June and 11500 for July. August cannot be taken into account since the campaign has been active during this month for only 2 days.  

The last analysis (cell 3) is not so relevant but we can notice a repeated pattern on the answers combinations, combinations that lead to most abandons ALWAYS contains a "no" answer, in my opinion people don't really identify to the topic and just leave the questionnaire, or some may not find it entertaining and so they leave after repeative negative answers.


In [ ]:
#Cell 1
#this cell shows the outcomes regarding the time of the day the session was completed and age groups of visitors.
from src.analytics.audience import (
    outcomes_by_age_and_time,
)
result = outcomes_by_age_and_time(session_path)

for age_group in ["16-24", "25-34", "35-44", "45-54", "55-64", "65+"]:
    print(f"\nAge group: {age_group}")
    display(
        result[result["age_group"] == age_group]
        [["session_period", "outcome_category", "total_sessions", "percentage"]]
    )

In [ ]:
#Cell 2
#this cell shows the monthly evolution of outcomes and sessions over time.
from src.analytics.campaign_performance import (
    monthly_outcome_evolution,
    monthly_session_evolution
)

display(monthly_outcome_evolution(session_path))
display(monthly_session_evolution(session_path))

In [ ]:
#cell 3
#this cell shows the most frequent combinations of questions that lead to dropouts in the questionnaire.
from src.analytics.questionnaire_analysis import (
    frequent_dropout_combinations
)

display(frequent_dropout_combinations(question_path))